# Local Photo ONNX Fixture Test

This notebook runs the exported ONNX model from `hallway_lighting_runs/exports/` on an uploaded hallway image.

It now does three things that matter for reliability:
- uses the same ImageNet normalization that the training pipeline used
- rejects near-black frames instead of trusting obviously invalid positive lux output
- estimates fixture count, under-fixture lux, and between-adjacent-fixture lux with an overlay

For best results, use an end-to-end hallway photo where the floor is visible in the lower part of the frame and the ceiling fixtures appear one after another down the corridor.


In [ ]:
%pip install onnxruntime pillow matplotlib ipywidgets numpy

In [ ]:
from pathlib import Path
import subprocess

REPO_URL = "https://github.com/Arwin-K/APS112-DL-Lighting-Camera.git"
COLAB_REPO_DIR = Path("/content/APS112-DL-Lighting-Camera")

if not COLAB_REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(COLAB_REPO_DIR)], check=True)
else:
    print(f"Using existing runtime clone: {COLAB_REPO_DIR}")


In [ ]:
from __future__ import annotations

from pathlib import Path
from typing import Any

import ipywidgets as widgets
from IPython.display import Markdown, clear_output, display
import matplotlib.pyplot as plt


PROJECT_ROOT_OVERRIDE: Path | None = None
MODEL_FILENAME = "hallway_multitask_unet_drive_prototype.onnx"
COLAB_PROJECT_ROOT = Path("/content/APS112-DL-Lighting-Camera/hallway_lighting")


def discover_notebook_project_root() -> Path:
    candidates: list[Path] = []
    if PROJECT_ROOT_OVERRIDE is not None:
        candidates.append(PROJECT_ROOT_OVERRIDE.expanduser())

    cwd = Path.cwd().resolve()
    candidates.extend([
        COLAB_PROJECT_ROOT,
        cwd,
        cwd / "hallway_lighting",
    ])

    current = cwd
    for _ in range(6):
        candidates.append(current)
        candidates.append(current / "hallway_lighting")
        if current.parent == current:
            break
        current = current.parent

    seen: set[Path] = set()
    ordered_candidates: list[Path] = []
    for candidate in candidates:
        try:
            resolved = candidate.resolve()
        except Exception:
            continue
        if resolved not in seen:
            seen.add(resolved)
            ordered_candidates.append(resolved)

    for candidate in ordered_candidates:
        model_path = candidate / "hallway_lighting_runs" / "exports" / MODEL_FILENAME
        if model_path.exists():
            return candidate

    raise FileNotFoundError(
        "Could not locate the hallway_lighting project folder and exported ONNX model. "
        "If you are in Colab, run the clone cell first. Otherwise open the notebook from inside the repo. "
        f"Candidates checked: {[str(path) for path in ordered_candidates]}"
    )


PROJECT_ROOT = discover_notebook_project_root()
DEFAULT_ONNX_PATH = PROJECT_ROOT / "hallway_lighting_runs" / "exports" / MODEL_FILENAME
OUTPUT_DIR = PROJECT_ROOT / "hallway_lighting_runs" / "local_photo_tests"

FIXTURE_DETECTION_SOURCE = '"""Automatic lighting-fixture detection and between-region analysis helpers."""\n\nfrom __future__ import annotations\n\nfrom dataclasses import asdict, dataclass\nfrom typing import Any, Sequence\n\nimport numpy as np\n\n@dataclass(frozen=True)\nclass FixtureDetection:\n    """Represents one detected ceiling fixture in normalized image coordinates."""\n\n    name: str\n    x: float\n    y: float\n    confidence: float\n    bbox: tuple[float, float, float, float]\n\n\n@dataclass(frozen=True)\nclass BetweenFixtureRegion:\n    """Represents a floor-plane region between two adjacent detected fixtures."""\n\n    name: str\n    left_fixture: str\n    right_fixture: str\n    polygon: list[tuple[float, float]]\n    area_pixels: int\n    area_ratio: float\n    estimated_area_m2: float | None\n\n\n@dataclass(frozen=True)\nclass FixturePointTarget:\n    """Point target compatible with the inference sampler interface."""\n\n    name: str\n    x: float\n    y: float\n    group: str\n    surface: str = "floor"\n\n\n@dataclass(frozen=True)\nclass FixtureLayout:\n    """Structured output for automatic fixture detection and floor projection."""\n\n    source: str\n    fallback_used: bool\n    search_region_bottom_y: float\n    floor_reference_y: float\n    fixtures: list[FixtureDetection]\n    point_targets: list[FixturePointTarget]\n    between_regions: list[BetweenFixtureRegion]\n\n    def to_summary_dict(self) -> dict[str, Any]:\n        """Returns a JSON-serializable representation."""\n\n        return {\n            "source": self.source,\n            "fallback_used": self.fallback_used,\n            "inferred_fixture_count": len(self.fixtures),\n            "search_region_bottom_y": self.search_region_bottom_y,\n            "floor_reference_y": self.floor_reference_y,\n            "fixtures": [asdict(fixture) for fixture in self.fixtures],\n            "point_targets": [asdict(point) for point in self.point_targets],\n            "between_regions": [asdict(region) for region in self.between_regions],\n        }\n\n\n@dataclass(frozen=True)\nclass CorridorGeometry:\n    """Simple hallway geometry derived from the floor mask."""\n\n    centerline_x: np.ndarray\n    left_edge_x: np.ndarray\n    right_edge_x: np.ndarray\n    floor_top_row: int\n    floor_bottom_row: int\n\n\ndef _ensure_rgb_float(image: np.ndarray) -> np.ndarray:\n    """Normalizes an image array to HxWx3 float space in [0, 1]."""\n\n    image_np = np.asarray(image)\n    if image_np.ndim != 3:\n        raise ValueError(f"Expected an HxWxC image, got shape {tuple(image_np.shape)}")\n    if image_np.shape[-1] == 1:\n        image_np = np.repeat(image_np, 3, axis=-1)\n    if image_np.shape[-1] != 3:\n        raise ValueError(f"Expected 3 image channels, got shape {tuple(image_np.shape)}")\n\n    image_np = image_np.astype(np.float32)\n    if image_np.max(initial=0.0) > 1.0 or image_np.min(initial=0.0) < 0.0:\n        image_np = image_np / 255.0\n    return np.clip(image_np, 0.0, 1.0)\n\n\ndef _normalize_score_map(values: np.ndarray) -> np.ndarray:\n    """Applies robust percentile normalization to a score map."""\n\n    values = np.asarray(values, dtype=np.float32)\n    low = float(np.percentile(values, 2))\n    high = float(np.percentile(values, 98))\n    if high <= low + 1e-6:\n        return np.clip(values, 0.0, 1.0)\n    return np.clip((values - low) / (high - low), 0.0, 1.0)\n\n\ndef _resolve_floor_mask(floor_mask: np.ndarray | None, height: int, width: int) -> np.ndarray | None:\n    """Normalizes an optional floor mask to a 2D boolean array."""\n\n    if floor_mask is None:\n        return None\n\n    floor_mask_np = np.asarray(floor_mask)\n    if floor_mask_np.ndim == 3:\n        if floor_mask_np.shape[0] == 1:\n            floor_mask_np = floor_mask_np[0]\n        elif floor_mask_np.shape[-1] == 1:\n            floor_mask_np = floor_mask_np[..., 0]\n        else:\n            floor_mask_np = floor_mask_np[..., 0]\n    if floor_mask_np.ndim != 2:\n        raise ValueError(f"Expected a 2D floor mask, got shape {tuple(floor_mask_np.shape)}")\n    if floor_mask_np.shape != (height, width):\n        raise ValueError(\n            f"Expected floor mask shape {(height, width)}, got {tuple(floor_mask_np.shape)}"\n        )\n    return floor_mask_np.astype(np.float32) > 0.5\n\n\ndef _estimate_search_region_bottom(floor_mask: np.ndarray | None, height: int) -> int:\n    """Chooses the lower boundary of the ceiling search region."""\n\n    default_bottom = max(1, min(height - 1, int(round(height * 0.72))))\n    if floor_mask is None or float(floor_mask.mean()) <= 0.01:\n        return default_bottom\n\n    floor_rows = np.where(floor_mask.any(axis=1))[0]\n    if floor_rows.size == 0:\n        return default_bottom\n\n    floor_top = int(floor_rows[0])\n    # Ignore clearly implausible floor masks that start too high in the image.\n    if floor_top < int(round(height * 0.30)):\n        return default_bottom\n\n    padded_bottom = floor_top - max(4, int(round(height * 0.04)))\n    min_reasonable_bottom = int(round(height * 0.42))\n    return max(1, min(default_bottom, max(min_reasonable_bottom, padded_bottom)))\n\n\ndef _gaussian_kernel1d(sigma: float) -> np.ndarray:\n    """Builds a normalized 1D Gaussian kernel."""\n\n    sigma = max(float(sigma), 0.5)\n    radius = max(1, int(round(sigma * 3.0)))\n    positions = np.arange(-radius, radius + 1, dtype=np.float32)\n    kernel = np.exp(-(positions**2) / (2.0 * sigma**2))\n    kernel_sum = float(kernel.sum())\n    if kernel_sum <= 1e-8:\n        return np.array([1.0], dtype=np.float32)\n    return kernel / kernel_sum\n\n\ndef _smooth_profile(profile: np.ndarray, sigma: float) -> np.ndarray:\n    """Smooths a 1D response profile."""\n\n    kernel = _gaussian_kernel1d(sigma)\n    return np.convolve(profile.astype(np.float32), kernel, mode="same")\n\n\ndef _build_fixture_score_map(image: np.ndarray) -> np.ndarray:\n    """Builds a heuristic score map for likely ceiling fixtures."""\n\n    rgb = _ensure_rgb_float(image)\n    luminance = 0.2126 * rgb[..., 0] + 0.7152 * rgb[..., 1] + 0.0722 * rgb[..., 2]\n    saturation = np.max(rgb, axis=-1) - np.min(rgb, axis=-1)\n\n    horizontal_detail = np.zeros(rgb.shape[:2], dtype=np.float32)\n    vertical_detail = np.zeros(rgb.shape[:2], dtype=np.float32)\n    horizontal_detail[:, 1:] = np.abs(rgb[:, 1:, :] - rgb[:, :-1, :]).mean(axis=-1)\n    vertical_detail[1:, :] = np.abs(rgb[1:, :, :] - rgb[:-1, :, :]).mean(axis=-1)\n    local_detail = np.clip(horizontal_detail + vertical_detail, 0.0, 1.0)\n\n    score = 0.70 * luminance + 0.20 * local_detail - 0.18 * saturation\n    return _normalize_score_map(score)\n\n\ndef _build_column_profile(score_map: np.ndarray, search_bottom: int) -> np.ndarray:\n    """Builds a smoothed horizontal response profile over the search region."""\n\n    ceiling_region = score_map[:search_bottom, :]\n    if ceiling_region.size == 0:\n        return np.zeros(score_map.shape[1], dtype=np.float32)\n\n    profile = (\n        0.55 * np.max(ceiling_region, axis=0)\n        + 0.30 * np.percentile(ceiling_region, 90, axis=0)\n        + 0.15 * np.mean(ceiling_region, axis=0)\n    )\n    sigma = max(1.0, score_map.shape[1] * 0.012)\n    return _smooth_profile(profile=profile, sigma=sigma)\n\n\ndef _detect_profile_peaks(\n    profile: np.ndarray,\n    min_horizontal_gap_px: int,\n    max_fixture_count: int,\n) -> tuple[list[int], bool]:\n    """Returns horizontal peak positions from the fixture profile."""\n\n    if profile.size == 0 or float(profile.max(initial=0.0)) <= 1e-6:\n        return [], False\n\n    min_height = max(0.08, float(np.percentile(profile, 80)))\n    candidate_indices: list[int] = []\n    for index in range(profile.size):\n        left = profile[index - 1] if index > 0 else profile[index]\n        right = profile[index + 1] if index + 1 < profile.size else profile[index]\n        if profile[index] >= min_height and profile[index] >= left and profile[index] >= right:\n            candidate_indices.append(index)\n\n    selected: list[int] = []\n    for index in sorted(candidate_indices, key=lambda item: float(profile[item]), reverse=True):\n        if all(abs(index - existing) >= max(1, min_horizontal_gap_px) for existing in selected):\n            selected.append(index)\n        if len(selected) >= max_fixture_count:\n            break\n\n    fallback_used = False\n    if len(selected) < max(1, min(3, max_fixture_count)):\n        fallback_used = True\n        suppressed_profile = profile.copy()\n        selected = []\n        absolute_floor = max(0.05, float(np.percentile(profile, 55)))\n        for _ in range(max_fixture_count):\n            peak_index = int(np.argmax(suppressed_profile))\n            peak_value = float(suppressed_profile[peak_index])\n            if peak_value < absolute_floor:\n                break\n            selected.append(peak_index)\n            left = max(0, peak_index - min_horizontal_gap_px)\n            right = min(profile.size, peak_index + min_horizontal_gap_px + 1)\n            suppressed_profile[left:right] = 0.0\n        selected.sort()\n\n    return selected, fallback_used\n\n\ndef _derive_peak_positions_from_rows(\n    score_map: np.ndarray,\n    search_bottom: int,\n    min_horizontal_gap_px: int,\n    max_fixture_count: int,\n) -> list[int]:\n    """Builds horizontal peak hypotheses from bright-row responses."""\n\n    ceiling_region = score_map[:search_bottom, :]\n    if ceiling_region.size == 0:\n        return []\n\n    row_scores = np.max(ceiling_region, axis=1)\n    if row_scores.size == 0:\n        return []\n\n    brightest_rows = np.argsort(row_scores)[::-1][: max(4, min(search_bottom, 12))]\n    candidate_columns: list[int] = []\n    min_value = max(0.08, float(np.percentile(ceiling_region, 88)))\n    for row_index in brightest_rows:\n        row = ceiling_region[int(row_index)]\n        for col_index in range(row.shape[0]):\n            left = row[col_index - 1] if col_index > 0 else row[col_index]\n            right = row[col_index + 1] if col_index + 1 < row.shape[0] else row[col_index]\n            if row[col_index] >= min_value and row[col_index] >= left and row[col_index] >= right:\n                candidate_columns.append(col_index)\n\n    if not candidate_columns:\n        return []\n\n    histogram = np.zeros(score_map.shape[1], dtype=np.float32)\n    for col_index in candidate_columns:\n        histogram[col_index] += 1.0\n    histogram = _smooth_profile(histogram, sigma=max(1.0, score_map.shape[1] * 0.01))\n\n    selected: list[int] = []\n    working = histogram.copy()\n    floor_value = max(0.5, float(np.percentile(histogram[histogram > 0], 40))) if np.any(histogram > 0) else 0.0\n    for _ in range(max_fixture_count):\n        peak_index = int(np.argmax(working))\n        peak_value = float(working[peak_index])\n        if peak_value < floor_value:\n            break\n        selected.append(peak_index)\n        left = max(0, peak_index - min_horizontal_gap_px)\n        right = min(working.size, peak_index + min_horizontal_gap_px + 1)\n        working[left:right] = 0.0\n    return sorted(selected)\n\n\ndef _estimate_floor_reference_y(\n    floor_mask: np.ndarray | None,\n    height: int,\n) -> float:\n    """Estimates a normalized floor sampling y-coordinate for point targets."""\n\n    if floor_mask is None or float(floor_mask.mean()) <= 0.01:\n        return 0.72\n\n    floor_rows = np.where(floor_mask.any(axis=1))[0]\n    if floor_rows.size == 0:\n        return 0.72\n\n    floor_top = float(floor_rows[0])\n    floor_bottom = float(floor_rows[-1])\n    sampling_y = floor_top + 0.18 * (floor_bottom - floor_top)\n    return float(np.clip(sampling_y / max(1, height - 1), 0.0, 1.0))\n\n\ndef _build_corridor_geometry(\n    floor_mask: np.ndarray | None,\n    height: int,\n    width: int,\n) -> CorridorGeometry:\n    """Builds centerline and floor-edge estimates for a hallway scene."""\n\n    row_indices = np.arange(height, dtype=np.float32)\n    default_center = np.full(height, (width - 1) / 2.0, dtype=np.float32)\n    default_left = np.full(height, width * 0.25, dtype=np.float32)\n    default_right = np.full(height, width * 0.75, dtype=np.float32)\n\n    if floor_mask is None or float(floor_mask.mean()) <= 0.01:\n        return CorridorGeometry(\n            centerline_x=default_center,\n            left_edge_x=default_left,\n            right_edge_x=default_right,\n            floor_top_row=int(round(height * 0.55)),\n            floor_bottom_row=height - 1,\n        )\n\n    floor_rows = np.where(floor_mask.any(axis=1))[0]\n    if floor_rows.size == 0:\n        return CorridorGeometry(\n            centerline_x=default_center,\n            left_edge_x=default_left,\n            right_edge_x=default_right,\n            floor_top_row=int(round(height * 0.55)),\n            floor_bottom_row=height - 1,\n        )\n\n    center_samples: list[float] = []\n    width_samples: list[float] = []\n    sample_rows: list[float] = []\n    for row_index in floor_rows:\n        columns = np.where(floor_mask[int(row_index)])[0]\n        if columns.size == 0:\n            continue\n        left = float(columns[0])\n        right = float(columns[-1])\n        center = (left + right) / 2.0\n        center_samples.append(center)\n        width_samples.append(max(1.0, right - left))\n        sample_rows.append(float(row_index))\n\n    if not sample_rows:\n        return CorridorGeometry(\n            centerline_x=default_center,\n            left_edge_x=default_left,\n            right_edge_x=default_right,\n            floor_top_row=int(round(height * 0.55)),\n            floor_bottom_row=height - 1,\n        )\n\n    rows_np = np.asarray(sample_rows, dtype=np.float32)\n    center_np = np.asarray(center_samples, dtype=np.float32)\n    width_np = np.asarray(width_samples, dtype=np.float32)\n\n    if rows_np.size >= 2:\n        center_coeff = np.polyfit(rows_np, center_np, deg=1)\n        centerline_x = np.polyval(center_coeff, row_indices).astype(np.float32)\n        width_coeff = np.polyfit(rows_np, width_np, deg=1)\n        fitted_width = np.polyval(width_coeff, row_indices).astype(np.float32)\n    else:\n        centerline_x = np.full(height, center_np[0], dtype=np.float32)\n        fitted_width = np.full(height, width_np[0], dtype=np.float32)\n\n    min_width = max(width * 0.10, float(np.percentile(width_np, 15)) * 0.75)\n    max_width = width * 0.95\n    fitted_width = np.clip(fitted_width, min_width, max_width)\n    left_edge_x = centerline_x - fitted_width / 2.0\n    right_edge_x = centerline_x + fitted_width / 2.0\n    centerline_x = np.clip(centerline_x, 0.0, width - 1.0)\n    left_edge_x = np.clip(left_edge_x, 0.0, width - 2.0)\n    right_edge_x = np.clip(right_edge_x, 1.0, width - 1.0)\n    right_edge_x = np.maximum(right_edge_x, left_edge_x + 1.0)\n    centerline_x = np.clip(centerline_x, left_edge_x, right_edge_x)\n\n    return CorridorGeometry(\n        centerline_x=centerline_x,\n        left_edge_x=left_edge_x,\n        right_edge_x=right_edge_x,\n        floor_top_row=int(floor_rows[0]),\n        floor_bottom_row=int(floor_rows[-1]),\n    )\n\n\ndef _floor_row_center_x(\n    row_index: int,\n    floor_mask: np.ndarray | None,\n    corridor: CorridorGeometry,\n) -> float:\n    """Returns the center x-position of the floor at a given row."""\n\n    if floor_mask is not None:\n        floor_columns = np.where(floor_mask[row_index])[0]\n        if floor_columns.size > 0:\n            return float((floor_columns[0] + floor_columns[-1]) / 2.0)\n    return float(corridor.centerline_x[row_index])\n\n\ndef _build_corridor_row_profile(\n    score_map: np.ndarray,\n    search_bottom: int,\n    corridor: CorridorGeometry,\n) -> np.ndarray:\n    """Builds a vertical response profile along the hallway centerline."""\n\n    height, width = score_map.shape\n    profile = np.zeros(height, dtype=np.float32)\n    for row_index in range(search_bottom):\n        center_x = float(corridor.centerline_x[row_index])\n        corridor_width = float(corridor.right_edge_x[row_index] - corridor.left_edge_x[row_index])\n        window_radius = int(round(max(width * 0.04, min(width * 0.16, corridor_width * 0.22))))\n        left = max(0, int(round(center_x)) - window_radius)\n        right = min(width, int(round(center_x)) + window_radius + 1)\n        if right <= left:\n            continue\n        strip = score_map[row_index, left:right]\n        if strip.size == 0:\n            continue\n\n        positions = np.arange(left, right, dtype=np.float32)\n        distance = np.abs(positions - center_x)\n        weights = np.clip(1.0 - distance / max(1.0, float(window_radius)), 0.15, 1.0)\n        weighted_strip = strip * weights\n        profile[row_index] = (\n            0.78 * float(np.max(weighted_strip))\n            + 0.14 * float(np.mean(weighted_strip))\n            + 0.08 * float(np.percentile(weighted_strip, 90))\n        )\n\n    sigma = max(1.0, search_bottom * 0.008)\n    return _smooth_profile(profile[:search_bottom], sigma=sigma)\n\n\ndef _detect_vertical_peaks(\n    profile: np.ndarray,\n    max_fixture_count: int,\n    min_vertical_gap_px: int,\n) -> tuple[list[int], bool]:\n    """Detects fixture rows along the corridor depth axis."""\n\n    if profile.size == 0 or float(profile.max(initial=0.0)) <= 1e-6:\n        return [], False\n\n    threshold = max(0.025, float(profile.max()) * 0.22, float(np.percentile(profile, 52)))\n    candidate_rows: list[int] = []\n    for row_index in range(profile.size):\n        left = profile[row_index - 1] if row_index > 0 else profile[row_index]\n        right = profile[row_index + 1] if row_index + 1 < profile.size else profile[row_index]\n        if profile[row_index] >= threshold and profile[row_index] >= left and profile[row_index] >= right:\n            candidate_rows.append(row_index)\n\n    selected: list[int] = []\n    for row_index in candidate_rows:\n        if not selected:\n            selected.append(row_index)\n            continue\n        if row_index - selected[-1] < min_vertical_gap_px:\n            if float(profile[row_index]) > float(profile[selected[-1]]):\n                selected[-1] = row_index\n            continue\n        selected.append(row_index)\n        if len(selected) >= max_fixture_count:\n            break\n\n    fallback_used = False\n    if len(selected) < max(1, min(3, max_fixture_count)):\n        fallback_used = True\n        working = profile.copy()\n        selected = []\n        floor_value = max(0.04, float(np.percentile(profile, 40)))\n        for _ in range(max_fixture_count):\n            peak_row = int(np.argmax(working))\n            peak_value = float(working[peak_row])\n            if peak_value < floor_value:\n                break\n            selected.append(peak_row)\n            top = max(0, peak_row - min_vertical_gap_px)\n            bottom = min(working.size, peak_row + min_vertical_gap_px + 1)\n            working[top:bottom] = 0.0\n\n    return sorted(selected), fallback_used\n\n\ndef _refine_fixture_position(\n    score_map: np.ndarray,\n    row_index: int,\n    corridor: CorridorGeometry,\n    search_bottom: int,\n) -> tuple[int, int, float]:\n    """Refines a fixture candidate around the corridor centerline."""\n\n    height, width = score_map.shape\n    center_x = float(corridor.centerline_x[row_index])\n    corridor_width = float(corridor.right_edge_x[row_index] - corridor.left_edge_x[row_index])\n    x_radius = int(round(max(width * 0.04, min(width * 0.16, corridor_width * 0.24))))\n    y_radius = max(3, int(round(search_bottom * 0.025)))\n\n    top = max(0, row_index - y_radius)\n    bottom = min(search_bottom, row_index + y_radius + 1)\n    left = max(0, int(round(center_x)) - x_radius)\n    right = min(width, int(round(center_x)) + x_radius + 1)\n    patch = score_map[top:bottom, left:right]\n    if patch.size == 0:\n        return int(round(center_x)), row_index, 0.0\n\n    patch_y, patch_x = np.unravel_index(int(np.argmax(patch)), patch.shape)\n    peak_y = top + int(patch_y)\n    peak_x = left + int(patch_x)\n    confidence = float(np.clip(score_map[peak_y, peak_x], 0.0, 1.0))\n    return peak_x, peak_y, confidence\n\n\ndef _project_fixture_points_to_floor(\n    fixtures: Sequence[FixtureDetection],\n    corridor: CorridorGeometry,\n    floor_mask: np.ndarray | None,\n    height: int,\n    width: int,\n) -> list[FixturePointTarget]:\n    """Projects fixtures onto the hallway floor along corridor depth."""\n\n    if not fixtures:\n        return []\n\n    search_bottom_row = min(corridor.floor_top_row - 1, int(round(height * 0.72)))\n    search_bottom_row = max(1, search_bottom_row)\n    floor_span = max(1.0, float(corridor.floor_bottom_row - corridor.floor_top_row))\n    source_span = max(1.0, float(search_bottom_row))\n\n    projected_rows: list[float] = []\n    for fixture in fixtures:\n        fixture_row = fixture.y * max(1, height - 1)\n        depth_ratio = np.clip(fixture_row / source_span, 0.0, 1.0)\n        projected_row = corridor.floor_top_row + (depth_ratio**1.35) * floor_span\n        projected_rows.append(float(projected_row))\n\n    ordered_rows = np.asarray(projected_rows, dtype=np.float32)\n    if ordered_rows.size > 1:\n        min_step = max(2.0, floor_span * 0.04)\n        for index in range(1, ordered_rows.size):\n            ordered_rows[index] = max(ordered_rows[index], ordered_rows[index - 1] + min_step)\n        ordered_rows = np.clip(ordered_rows, corridor.floor_top_row, corridor.floor_bottom_row)\n\n    def snap_floor_point(projected_row: float) -> tuple[float, float]:\n        row_index = int(np.clip(round(projected_row), corridor.floor_top_row, corridor.floor_bottom_row))\n        center_x = float(corridor.centerline_x[row_index])\n\n        if floor_mask is None:\n            return (\n                float(np.clip(center_x / max(1, width - 1), 0.0, 1.0)),\n                float(np.clip(projected_row / max(1, height - 1), 0.0, 1.0)),\n            )\n\n        search_order = [row_index]\n        max_offset = max(1, int(round(height * 0.03)))\n        for offset in range(1, max_offset + 1):\n            below = min(corridor.floor_bottom_row, row_index + offset)\n            above = max(corridor.floor_top_row, row_index - offset)\n            if below not in search_order:\n                search_order.append(below)\n            if above not in search_order:\n                search_order.append(above)\n\n        for candidate_row in search_order:\n            floor_columns = np.where(floor_mask[candidate_row])[0]\n            if floor_columns.size == 0:\n                continue\n            snapped_x = float((floor_columns[0] + floor_columns[-1]) / 2.0)\n            return (\n                float(np.clip(snapped_x / max(1, width - 1), 0.0, 1.0)),\n                float(np.clip(candidate_row / max(1, height - 1), 0.0, 1.0)),\n            )\n\n        return (\n            float(np.clip(center_x / max(1, width - 1), 0.0, 1.0)),\n            float(np.clip(projected_row / max(1, height - 1), 0.0, 1.0)),\n        )\n\n    point_targets: list[FixturePointTarget] = []\n    for index, projected_row in enumerate(ordered_rows, start=1):\n        point_x, point_y = snap_floor_point(float(projected_row))\n        point_targets.append(\n            FixturePointTarget(\n                name=f"under_fixture_{index}",\n                x=point_x,\n                y=point_y,\n                group="under_fixture",\n            )\n        )\n\n    for index in range(len(point_targets) - 1):\n        left_row = point_targets[index].y * max(1, height - 1)\n        right_row = point_targets[index + 1].y * max(1, height - 1)\n        midpoint_row = (left_row + right_row) / 2.0\n        point_x, point_y = snap_floor_point(float(midpoint_row))\n        point_targets.append(\n            FixturePointTarget(\n                name=f"between_fixture_{index + 1}_{index + 2}",\n                x=point_x,\n                y=point_y,\n                group="between_fixture",\n            )\n        )\n    return point_targets\n\n\ndef _local_peak_y(score_map: np.ndarray, search_bottom: int, peak_x: int, window_radius: int) -> int:\n    """Finds the best vertical position for a fixture near a horizontal peak."""\n\n    left = max(0, peak_x - window_radius)\n    right = min(score_map.shape[1], peak_x + window_radius + 1)\n    local_region = score_map[:search_bottom, left:right]\n    if local_region.size == 0:\n        return max(0, min(search_bottom - 1, int(round(search_bottom * 0.25))))\n    y_index, _ = np.unravel_index(int(np.argmax(local_region)), local_region.shape)\n    return int(y_index)\n\n\ndef _bbox_from_peak(\n    score_map: np.ndarray,\n    peak_x: int,\n    peak_y: int,\n    search_bottom: int,\n    window_radius: int,\n) -> tuple[float, float, float, float]:\n    """Builds a rough bounding box around a detected fixture peak."""\n\n    height, width = score_map.shape\n    local_peak = float(score_map[peak_y, peak_x])\n    threshold = max(0.45, local_peak * 0.65)\n\n    min_col = peak_x\n    max_col = peak_x\n    while min_col > 0 and score_map[peak_y, min_col - 1] >= threshold:\n        min_col -= 1\n    while max_col + 1 < width and score_map[peak_y, max_col + 1] >= threshold:\n        max_col += 1\n\n    min_row = peak_y\n    max_row = peak_y\n    while min_row > 0 and score_map[min_row - 1, peak_x] >= threshold:\n        min_row -= 1\n    while max_row + 1 < search_bottom and score_map[max_row + 1, peak_x] >= threshold:\n        max_row += 1\n\n    if max_col == min_col:\n        min_col = max(0, peak_x - window_radius // 2)\n        max_col = min(width - 1, peak_x + window_radius // 2)\n    if max_row == min_row:\n        min_row = max(0, peak_y - window_radius // 3)\n        max_row = min(search_bottom - 1, peak_y + window_radius // 3)\n\n    return (\n        float(np.clip(min_col / max(1, width - 1), 0.0, 1.0)),\n        float(np.clip(min_row / max(1, height - 1), 0.0, 1.0)),\n        float(np.clip(max_col / max(1, width - 1), 0.0, 1.0)),\n        float(np.clip(max_row / max(1, height - 1), 0.0, 1.0)),\n    )\n\n\ndef _project_points_to_floor(\n    fixtures: Sequence[FixtureDetection],\n    floor_reference_y: float,\n) -> list[FixturePointTarget]:\n    """Creates under- and between-fixture floor point targets."""\n\n    point_targets: list[FixturePointTarget] = []\n    for index, fixture in enumerate(fixtures, start=1):\n        point_targets.append(\n            FixturePointTarget(\n                name=f"under_fixture_{index}",\n                x=fixture.x,\n                y=floor_reference_y,\n                group="under_fixture",\n            )\n        )\n\n    for index, (left, right) in enumerate(zip(fixtures[:-1], fixtures[1:]), start=1):\n        point_targets.append(\n            FixturePointTarget(\n                name=f"between_fixture_{index}_{index + 1}",\n                x=(left.x + right.x) / 2.0,\n                y=floor_reference_y,\n                group="between_fixture",\n            )\n        )\n    return point_targets\n\n\ndef _build_between_regions(\n    point_targets: Sequence[FixturePointTarget],\n    corridor: CorridorGeometry,\n    floor_mask: np.ndarray | None,\n    floor_area_m2: float | None,\n    height: int,\n    width: int,\n) -> list[BetweenFixtureRegion]:\n    """Creates simple floor-plane regions between adjacent projected points."""\n\n    under_points = [point for point in point_targets if point.group == "under_fixture"]\n    if len(under_points) < 2:\n        return []\n    floor_pixels = int(floor_mask.sum()) if floor_mask is not None else 0\n\n    regions: list[BetweenFixtureRegion] = []\n    for index, (left, right) in enumerate(zip(under_points[:-1], under_points[1:]), start=1):\n        top_row = int(round(left.y * max(1, height - 1)))\n        bottom_row = int(round(right.y * max(1, height - 1)))\n        if bottom_row < top_row:\n            top_row, bottom_row = bottom_row, top_row\n\n        top_left_x = float(corridor.left_edge_x[top_row])\n        top_right_x = float(corridor.right_edge_x[top_row])\n        bottom_left_x = float(corridor.left_edge_x[bottom_row])\n        bottom_right_x = float(corridor.right_edge_x[bottom_row])\n        region_mask = np.zeros((height, width), dtype=bool)\n        for row_index in range(top_row, bottom_row + 1):\n            left_x = int(round(corridor.left_edge_x[row_index]))\n            right_x = int(round(corridor.right_edge_x[row_index]))\n            region_mask[row_index, left_x : right_x + 1] = True\n        if floor_mask is not None:\n            region_mask &= floor_mask\n\n        area_pixels = int(region_mask.sum())\n        area_ratio = float(area_pixels / float(height * width))\n        estimated_area_m2 = None\n        if floor_area_m2 is not None and floor_pixels > 0:\n            estimated_area_m2 = float(floor_area_m2 * (area_pixels / float(floor_pixels)))\n\n        polygon = [\n            (\n                float(np.clip(top_left_x / max(1, width - 1), 0.0, 1.0)),\n                float(np.clip(top_row / max(1, height - 1), 0.0, 1.0)),\n            ),\n            (\n                float(np.clip(top_right_x / max(1, width - 1), 0.0, 1.0)),\n                float(np.clip(top_row / max(1, height - 1), 0.0, 1.0)),\n            ),\n            (\n                float(np.clip(bottom_right_x / max(1, width - 1), 0.0, 1.0)),\n                float(np.clip(bottom_row / max(1, height - 1), 0.0, 1.0)),\n            ),\n            (\n                float(np.clip(bottom_left_x / max(1, width - 1), 0.0, 1.0)),\n                float(np.clip(bottom_row / max(1, height - 1), 0.0, 1.0)),\n            ),\n        ]\n        regions.append(\n            BetweenFixtureRegion(\n                name=f"between_fixture_{index}_{index + 1}",\n                left_fixture=f"fixture_{index}",\n                right_fixture=f"fixture_{index + 1}",\n                polygon=polygon,\n                area_pixels=area_pixels,\n                area_ratio=area_ratio,\n                estimated_area_m2=estimated_area_m2,\n            )\n        )\n    return regions\n\n\ndef infer_fixture_layout(\n    image: np.ndarray,\n    floor_mask: np.ndarray | None = None,\n    min_fixture_count: int = 1,\n    max_fixture_count: int = 8,\n    min_horizontal_gap_ratio: float = 0.08,\n    floor_area_m2: float | None = None,\n) -> FixtureLayout | None:\n    """Infers fixture count, fixture positions, and between-fixture regions from an image."""\n\n    rgb = _ensure_rgb_float(image)\n    height, width = rgb.shape[:2]\n    floor_mask_np = _resolve_floor_mask(floor_mask, height=height, width=width)\n    search_bottom = _estimate_search_region_bottom(floor_mask_np, height=height)\n    if search_bottom <= 1:\n        return None\n\n    corridor = _build_corridor_geometry(floor_mask_np, height=height, width=width)\n    score_map = _build_fixture_score_map(rgb)\n    score_map[search_bottom:, :] = 0.0\n    profile = _build_corridor_row_profile(\n        score_map=score_map,\n        search_bottom=search_bottom,\n        corridor=corridor,\n    )\n    min_vertical_gap_px = max(5, int(round(height * 0.035)))\n    peak_rows, fallback_used = _detect_vertical_peaks(\n        profile=profile,\n        max_fixture_count=max_fixture_count,\n        min_vertical_gap_px=min_vertical_gap_px,\n    )\n    if len(peak_rows) < min_fixture_count:\n        return None\n\n    fixtures: list[FixtureDetection] = []\n    for index, peak_row in enumerate(peak_rows, start=1):\n        peak_x, peak_y, confidence = _refine_fixture_position(\n            score_map=score_map,\n            row_index=peak_row,\n            corridor=corridor,\n            search_bottom=search_bottom,\n        )\n        fixtures.append(\n            FixtureDetection(\n                name=f"fixture_{index}",\n                x=float(np.clip(peak_x / max(1, width - 1), 0.0, 1.0)),\n                y=float(np.clip(peak_y / max(1, height - 1), 0.0, 1.0)),\n                confidence=confidence,\n                bbox=_bbox_from_peak(\n                    score_map=score_map,\n                    peak_x=peak_x,\n                    peak_y=peak_y,\n                    search_bottom=search_bottom,\n                    window_radius=max(3, int(round(width * 0.04))),\n                ),\n            )\n        )\n\n    point_targets = _project_fixture_points_to_floor(\n        fixtures=fixtures,\n        corridor=corridor,\n        floor_mask=floor_mask_np,\n        height=height,\n        width=width,\n    )\n    if not point_targets:\n        return None\n\n    under_points = [point for point in point_targets if point.group == "under_fixture"]\n    floor_reference_y = under_points[0].y if under_points else _estimate_floor_reference_y(floor_mask_np, height=height)\n    between_regions = _build_between_regions(\n        point_targets=point_targets,\n        corridor=corridor,\n        floor_mask=floor_mask_np,\n        floor_area_m2=floor_area_m2,\n        height=height,\n        width=width,\n    )\n    return FixtureLayout(\n        source="automatic_corridor_fixture_detector",\n        fallback_used=fallback_used,\n        search_region_bottom_y=float(search_bottom / max(1, height - 1)),\n        floor_reference_y=floor_reference_y,\n        fixtures=fixtures,\n        point_targets=point_targets,\n        between_regions=between_regions,\n    )\n'
exec(FIXTURE_DETECTION_SOURCE, globals())

HELPER_SOURCE = '"""Shared helper code embedded into the local ONNX notebook."""\n\nfrom __future__ import annotations\n\nfrom io import BytesIO\nimport json\nfrom pathlib import Path\nfrom typing import Any\n\nimport matplotlib.pyplot as plt\nfrom matplotlib.path import Path as MplPath\nfrom matplotlib.patches import Polygon\nimport numpy as np\nfrom PIL import Image\n\nMODEL_FILENAME = "hallway_multitask_unet_drive_prototype.onnx"\nCOLAB_PROJECT_ROOT = Path("/content/APS112-DL-Lighting-Camera/hallway_lighting")\nIMAGENET_MEAN = np.asarray([0.485, 0.456, 0.406], dtype=np.float32).reshape(1, 1, 3)\nIMAGENET_STD = np.asarray([0.229, 0.224, 0.225], dtype=np.float32).reshape(1, 1, 3)\nDARK_FRAME_MEAN_THRESHOLD = 0.03\nDARK_FRAME_P95_THRESHOLD = 0.08\nEXPECTED_ONNX_OUTPUT_NAMES = [\n    "lux_map",\n    "avg_lux",\n    "low_lux_p5",\n    "high_lux_p95",\n    "floor_mask_pred",\n    "albedo_pred",\n    "gloss_pred",\n    "uncertainty_map",\n    "estimated_power_w",\n]\n\n\ndef load_onnx_session(onnx_path: Path):\n    """Loads the ONNX runtime session on CPU."""\n\n    try:\n        import onnxruntime as ort\n    except ImportError as exc:\n        raise ImportError(\n            "onnxruntime is not installed in this notebook environment. "\n            "Run the install cell, restart the kernel, and rerun the notebook."\n        ) from exc\n\n    if not onnx_path.exists():\n        raise FileNotFoundError(f"ONNX model not found: {onnx_path}")\n    return ort.InferenceSession(str(onnx_path), providers=["CPUExecutionProvider"])\n\n\ndef extract_single_map(value: np.ndarray | float | int | None) -> np.ndarray | None:\n    """Returns a single 2D or HxWxC array from an ONNX output value."""\n\n    if value is None or isinstance(value, (float, int)):\n        return None\n    array = np.asarray(value)\n    if array.ndim == 4:\n        array = array[0]\n    if array.ndim == 3 and array.shape[0] in {1, 3}:\n        array = np.transpose(array, (1, 2, 0))\n    if array.ndim == 3 and array.shape[-1] == 1:\n        array = array[..., 0]\n    return array.astype(np.float32, copy=False)\n\n\ndef extract_scalar(value: np.ndarray | float | int | None, fallback: float = 0.0) -> float:\n    """Extracts a scalar from a tensor-like output."""\n\n    if value is None:\n        return fallback\n    if isinstance(value, (float, int)):\n        return float(value)\n    array = np.asarray(value).reshape(-1)\n    return fallback if array.size == 0 else float(array[0])\n\n\ndef assess_image_quality(display_rgb: np.ndarray) -> dict[str, float | bool]:\n    """Computes simple brightness metrics and a dark-frame gate."""\n\n    rgb = np.clip(np.asarray(display_rgb, dtype=np.float32), 0.0, 1.0)\n    luminance = 0.2126 * rgb[..., 0] + 0.7152 * rgb[..., 1] + 0.0722 * rgb[..., 2]\n    bottom_start = min(rgb.shape[0] - 1, max(0, int(round(rgb.shape[0] * 0.55))))\n    bottom_band = luminance[bottom_start:, :] if bottom_start < rgb.shape[0] else luminance\n\n    mean_luminance = float(np.mean(luminance))\n    p95_luminance = float(np.percentile(luminance, 95))\n    max_luminance = float(np.max(luminance))\n    bottom_mean_luminance = float(np.mean(bottom_band))\n    is_dark_frame = (\n        mean_luminance < DARK_FRAME_MEAN_THRESHOLD\n        and p95_luminance < DARK_FRAME_P95_THRESHOLD\n    )\n    return {\n        "mean_luminance": mean_luminance,\n        "p95_luminance": p95_luminance,\n        "max_luminance": max_luminance,\n        "bottom_mean_luminance": bottom_mean_luminance,\n        "is_dark_frame": bool(is_dark_frame),\n    }\n\n\ndef preprocess_uploaded_image(\n    image_bytes: bytes,\n    input_height: int,\n    input_width: int,\n) -> tuple[np.ndarray, np.ndarray, dict[str, float | bool]]:\n    """Resizes image, keeps a display copy, and applies training-time normalization."""\n\n    image = Image.open(BytesIO(image_bytes)).convert("RGB")\n    resized = image.resize((input_width, input_height), resample=Image.BILINEAR)\n    display_rgb = np.asarray(resized).astype(np.float32) / 255.0\n    quality = assess_image_quality(display_rgb)\n    normalized_rgb = (display_rgb - IMAGENET_MEAN) / IMAGENET_STD\n    model_input = np.transpose(normalized_rgb, (2, 0, 1))[None, ...].astype(np.float32)\n    return model_input, display_rgb, quality\n\n\ndef summarize_lux_map(lux_map: np.ndarray, floor_mask: np.ndarray | None = None) -> dict[str, float]:\n    """Computes basic lux summary statistics."""\n\n    values = np.asarray(lux_map, dtype=np.float32)\n    if floor_mask is not None:\n        mask = np.asarray(floor_mask).astype(bool)\n        if mask.shape == values.shape and float(mask.mean()) > 0.001:\n            values = values[mask]\n        else:\n            values = values.reshape(-1)\n    else:\n        values = values.reshape(-1)\n\n    if values.size == 0:\n        return {"avg_lux": 0.0, "low_lux_p5": 0.0, "high_lux_p95": 0.0}\n    return {\n        "avg_lux": float(np.mean(values)),\n        "low_lux_p5": float(np.percentile(values, 5)),\n        "high_lux_p95": float(np.percentile(values, 95)),\n    }\n\n\ndef _sample_mask_mean(lux_map: np.ndarray, mask: np.ndarray) -> float:\n    mask_bool = np.asarray(mask, dtype=bool)\n    if mask_bool.shape != lux_map.shape:\n        raise ValueError("Sampling mask must match lux map shape.")\n    if not np.any(mask_bool):\n        return float("nan")\n    return float(np.mean(np.asarray(lux_map, dtype=np.float32)[mask_bool]))\n\n\ndef _build_disc_mask(\n    height: int,\n    width: int,\n    center_x: float,\n    center_y: float,\n    radius_x: float,\n    radius_y: float,\n) -> np.ndarray:\n    y_coords, x_coords = np.ogrid[:height, :width]\n    norm_x = (x_coords - center_x) / max(radius_x, 1.0)\n    norm_y = (y_coords - center_y) / max(radius_y, 1.0)\n    return (norm_x * norm_x + norm_y * norm_y) <= 1.0\n\n\ndef _polygon_mask(\n    polygon_points: list[tuple[float, float]],\n    height: int,\n    width: int,\n) -> np.ndarray:\n    if not polygon_points:\n        return np.zeros((height, width), dtype=bool)\n    polygon_pixels = np.asarray(\n        [\n            (float(x_value) * (width - 1), float(y_value) * (height - 1))\n            for x_value, y_value in polygon_points\n        ],\n        dtype=np.float32,\n    )\n    grid_x, grid_y = np.meshgrid(np.arange(width), np.arange(height))\n    points = np.stack([grid_x.ravel(), grid_y.ravel()], axis=-1)\n    mask = MplPath(polygon_pixels).contains_points(points, radius=0.5)\n    return mask.reshape(height, width)\n\n\ndef _build_under_fixture_mask(\n    point: dict[str, Any],\n    floor_mask: np.ndarray | None,\n    height: int,\n    width: int,\n) -> np.ndarray:\n    center_x = float(point["x"]) * (width - 1)\n    center_y = float(point["y"]) * (height - 1)\n\n    if floor_mask is not None:\n        row_index = int(np.clip(round(center_y), 0, height - 1))\n        floor_columns = np.where(floor_mask[row_index])[0]\n        floor_width = float(floor_columns[-1] - floor_columns[0]) if floor_columns.size > 1 else width * 0.2\n        radius_x = max(4.0, floor_width * 0.10)\n    else:\n        radius_x = max(4.0, width * 0.06)\n    radius_y = max(3.0, height * 0.02)\n\n    mask = _build_disc_mask(height, width, center_x, center_y, radius_x, radius_y)\n    if floor_mask is not None:\n        mask &= floor_mask\n    return mask\n\n\ndef compute_floor_measurements(\n    lux_map: np.ndarray,\n    floor_mask: np.ndarray | None,\n    fixture_analysis: dict[str, Any] | None,\n) -> tuple[dict[str, float], dict[str, float], dict[str, np.ndarray]]:\n    """Computes floor-area lux summaries under and between fixtures."""\n\n    if fixture_analysis is None:\n        return {}, {}, {}\n\n    height, width = lux_map.shape\n    point_targets = fixture_analysis.get("point_targets") or []\n    between_regions = fixture_analysis.get("between_regions") or []\n\n    measurement_masks: dict[str, np.ndarray] = {}\n    under_fixture_lux: dict[str, float] = {}\n    between_fixture_lux: dict[str, float] = {}\n\n    for point in point_targets:\n        point_name = str(point["name"])\n        if point.get("group") != "under_fixture":\n            continue\n        mask = _build_under_fixture_mask(point, floor_mask=floor_mask, height=height, width=width)\n        measurement_masks[point_name] = mask\n        under_fixture_lux[point_name] = _sample_mask_mean(lux_map, mask)\n\n    for region in between_regions:\n        region_name = str(region["name"])\n        mask = _polygon_mask(region.get("polygon") or [], height=height, width=width)\n        if floor_mask is not None:\n            mask &= floor_mask\n        measurement_masks[region_name] = mask\n        between_fixture_lux[region_name] = _sample_mask_mean(lux_map, mask)\n\n    return under_fixture_lux, between_fixture_lux, measurement_masks\n\n\ndef _find_measurement_value(\n    point_name: str,\n    under_fixture_lux: dict[str, float],\n    between_fixture_lux: dict[str, float],\n) -> float | None:\n    if point_name in under_fixture_lux:\n        return float(under_fixture_lux[point_name])\n    if point_name in between_fixture_lux:\n        return float(between_fixture_lux[point_name])\n    return None\n\n\ndef build_overlay_figure(\n    display_rgb: np.ndarray,\n    lux_map: np.ndarray,\n    fixture_analysis: dict[str, Any] | None,\n    under_fixture_lux: dict[str, float],\n    between_fixture_lux: dict[str, float],\n    measurement_masks: dict[str, np.ndarray] | None = None,\n    warning_texts: list[str] | None = None,\n):\n    """Builds the inline/saveable notebook overlay."""\n\n    figure, axis = plt.subplots(1, 1, figsize=(8, 5))\n    axis.imshow(display_rgb)\n    if float(np.max(lux_map, initial=0.0)) > 1e-6:\n        axis.imshow(lux_map, cmap="inferno", alpha=0.38)\n\n    height, width = display_rgb.shape[:2]\n    warning_texts = warning_texts or []\n    measurement_masks = measurement_masks or {}\n\n    if fixture_analysis is not None:\n        for region in fixture_analysis.get("between_regions", []):\n            polygon_points = region.get("polygon") or []\n            if polygon_points:\n                polygon_pixels = [\n                    (float(x_value) * (width - 1), float(y_value) * (height - 1))\n                    for x_value, y_value in polygon_points\n                ]\n                axis.add_patch(\n                    Polygon(\n                        polygon_pixels,\n                        closed=True,\n                        facecolor="#80cbc4",\n                        edgecolor="#004d40",\n                        linewidth=1.2,\n                        alpha=0.10,\n                    )\n                )\n\n        for fixture in fixture_analysis.get("fixtures", []):\n            x_pixel = float(fixture["x"]) * (width - 1)\n            y_pixel = float(fixture["y"]) * (height - 1)\n            axis.scatter(\n                x_pixel,\n                y_pixel,\n                s=120,\n                c="#ffee58",\n                edgecolors="black",\n                linewidths=0.9,\n                zorder=3,\n            )\n            axis.text(\n                x_pixel + 6,\n                y_pixel - 6,\n                str(fixture["name"]),\n                color="white",\n                fontsize=8,\n                bbox={"facecolor": "black", "alpha": 0.45, "pad": 2},\n            )\n\n        for point in fixture_analysis.get("point_targets", []):\n            point_name = str(point["name"])\n            measurement_value = _find_measurement_value(\n                point_name,\n                under_fixture_lux=under_fixture_lux,\n                between_fixture_lux=between_fixture_lux,\n            )\n            if measurement_value is None or not np.isfinite(measurement_value):\n                continue\n            x_pixel = float(point["x"]) * (width - 1)\n            y_pixel = float(point["y"]) * (height - 1)\n            point_color = "#4fc3f7" if point_name.startswith("between_") else "#ffffff"\n\n            mask = measurement_masks.get(point_name)\n            if mask is not None and np.any(mask):\n                mask_alpha = np.zeros((height, width, 4), dtype=np.float32)\n                rgba = np.array([0.20, 0.80, 1.00, 0.12], dtype=np.float32)\n                if point_name.startswith("under_"):\n                    rgba = np.array([1.00, 1.00, 1.00, 0.10], dtype=np.float32)\n                mask_alpha[mask] = rgba\n                axis.imshow(mask_alpha)\n\n            axis.scatter(\n                x_pixel,\n                y_pixel,\n                s=48,\n                c=point_color,\n                edgecolors="black",\n                linewidths=0.8,\n                zorder=4,\n            )\n            axis.text(\n                x_pixel + 4,\n                y_pixel + 10,\n                f"floor {point_name}\\n{measurement_value:.1f} lux",\n                color="white",\n                fontsize=7,\n                bbox={"facecolor": "black", "alpha": 0.5, "pad": 2},\n                zorder=5,\n            )\n\n    if warning_texts:\n        axis.text(\n            0.01,\n            0.99,\n            "\\n".join(warning_texts),\n            transform=axis.transAxes,\n            ha="left",\n            va="top",\n            color="white",\n            fontsize=8,\n            bbox={"facecolor": "#5d4037", "alpha": 0.72, "pad": 4},\n            zorder=6,\n        )\n\n    axis.set_title("Fixture-aware Floor Lux Overlay")\n    axis.axis("off")\n    figure.tight_layout()\n    return figure\n\n\ndef save_result_artifacts(\n    output_dir: Path,\n    image_name: str,\n    display_rgb: np.ndarray,\n    lux_map: np.ndarray,\n    fixture_analysis: dict[str, Any] | None,\n    under_fixture_lux: dict[str, float],\n    between_fixture_lux: dict[str, float],\n    measurement_masks: dict[str, np.ndarray],\n    result: dict[str, Any],\n    warning_texts: list[str],\n):\n    """Saves the overlay and JSON summary for a notebook run."""\n\n    figure = build_overlay_figure(\n        display_rgb=display_rgb,\n        lux_map=lux_map,\n        fixture_analysis=fixture_analysis,\n        under_fixture_lux=under_fixture_lux,\n        between_fixture_lux=between_fixture_lux,\n        measurement_masks=measurement_masks,\n        warning_texts=warning_texts,\n    )\n    output_dir.mkdir(parents=True, exist_ok=True)\n    image_stem = Path(image_name).stem\n    overlay_path = output_dir / f"{image_stem}_fixture_lux_overlay.png"\n    summary_path = output_dir / f"{image_stem}_fixture_lux_summary.json"\n    figure.savefig(overlay_path, bbox_inches="tight")\n    result["overlay_image"] = str(overlay_path)\n    result["summary_json"] = str(summary_path)\n    with summary_path.open("w", encoding="utf-8") as handle:\n        json.dump(result, handle, indent=2)\n    return figure\n\n\ndef _run_named_outputs(session, model_input: np.ndarray) -> dict[str, Any]:\n    raw_outputs = session.run(None, {session.get_inputs()[0].name: model_input})\n    session_output_names = [output.name for output in session.get_outputs()]\n    if session_output_names and len(session_output_names) == len(raw_outputs):\n        return {name: value for name, value in zip(session_output_names, raw_outputs)}\n    return {name: value for name, value in zip(EXPECTED_ONNX_OUTPUT_NAMES, raw_outputs)}\n\n\ndef run_uploaded_photo(\n    image_bytes: bytes,\n    image_name: str,\n    onnx_path: Path,\n    output_dir: Path,\n    max_fixture_count: int = 8,\n    floor_area_m2: float = 12.0,\n):\n    """Runs ONNX inference plus fixture-aware post-processing for the notebook."""\n\n    session = load_onnx_session(onnx_path)\n    input_shape = session.get_inputs()[0].shape\n    input_height = int(input_shape[2])\n    input_width = int(input_shape[3])\n    model_input, display_rgb, image_quality = preprocess_uploaded_image(\n        image_bytes=image_bytes,\n        input_height=input_height,\n        input_width=input_width,\n    )\n\n    warning_texts: list[str] = []\n    if bool(image_quality["is_dark_frame"]):\n        warning_texts.append(\n            "Frame is too dark for reliable lux inference. Returning 0 lux and no fixtures."\n        )\n        result = {\n            "image_name": image_name,\n            "onnx_path": str(onnx_path),\n            "fixture_count": 0,\n            "avg_lux": 0.0,\n            "low_lux_p5": 0.0,\n            "high_lux_p95": 0.0,\n            "lux_map_summary": {"avg_lux": 0.0, "low_lux_p5": 0.0, "high_lux_p95": 0.0},\n            "under_fixture_lux": {},\n            "between_fixture_lux": {},\n            "point_lux": {},\n            "fixture_analysis": None,\n            "warnings": warning_texts,\n            "image_quality": image_quality,\n            "inference_skipped": True,\n        }\n        zero_lux_map = np.zeros((input_height, input_width), dtype=np.float32)\n        figure = save_result_artifacts(\n            output_dir=output_dir,\n            image_name=image_name,\n            display_rgb=display_rgb,\n            lux_map=zero_lux_map,\n            fixture_analysis=None,\n            under_fixture_lux={},\n            between_fixture_lux={},\n            measurement_masks={},\n            result=result,\n            warning_texts=warning_texts,\n        )\n        return result, figure\n\n    named_outputs = _run_named_outputs(session, model_input=model_input)\n\n    lux_map = extract_single_map(named_outputs.get("lux_map"))\n    if lux_map is None or lux_map.ndim != 2:\n        raise ValueError("Expected a single-channel lux map from the ONNX output.")\n\n    floor_mask = extract_single_map(named_outputs.get("floor_mask_pred"))\n    floor_mask_binary = None if floor_mask is None else (np.asarray(floor_mask) > 0.5)\n\n    floor_mask_ratio = None\n    if floor_mask_binary is not None:\n        floor_mask_ratio = float(np.mean(floor_mask_binary))\n        if floor_mask_ratio < 0.03:\n            warning_texts.append(\n                "Predicted floor area is very small in this frame, so floor measurements may be unreliable."\n            )\n\n    fixture_layout = infer_fixture_layout(\n        image=display_rgb,\n        floor_mask=floor_mask_binary,\n        max_fixture_count=max_fixture_count,\n        floor_area_m2=floor_area_m2,\n    )\n    fixture_analysis = None if fixture_layout is None else fixture_layout.to_summary_dict()\n\n    under_fixture_lux, between_fixture_lux, measurement_masks = compute_floor_measurements(\n        lux_map=lux_map,\n        floor_mask=floor_mask_binary,\n        fixture_analysis=fixture_analysis,\n    )\n    point_lux = {**under_fixture_lux, **between_fixture_lux}\n\n    if fixture_analysis is None or int(fixture_analysis.get("inferred_fixture_count", 0)) == 0:\n        warning_texts.append(\n            "No fixtures were found in this frame. Use an end-to-end hallway image with visible floor "\n            "and increase Max fixtures if needed."\n        )\n    elif int(fixture_analysis.get("inferred_fixture_count", 0)) >= int(max_fixture_count):\n        warning_texts.append(\n            "Detected fixture count reached the Max fixtures cap. Increase that value if your hallway has more fixtures."\n        )\n\n    lux_summary = summarize_lux_map(lux_map=lux_map, floor_mask=floor_mask_binary)\n    result = {\n        "image_name": image_name,\n        "onnx_path": str(onnx_path),\n        "fixture_count": 0 if fixture_analysis is None else int(fixture_analysis.get("inferred_fixture_count", 0)),\n        "avg_lux": extract_scalar(named_outputs.get("avg_lux"), fallback=lux_summary["avg_lux"]),\n        "low_lux_p5": extract_scalar(named_outputs.get("low_lux_p5"), fallback=lux_summary["low_lux_p5"]),\n        "high_lux_p95": extract_scalar(\n            named_outputs.get("high_lux_p95"),\n            fallback=lux_summary["high_lux_p95"],\n        ),\n        "lux_map_summary": lux_summary,\n        "under_fixture_lux": under_fixture_lux,\n        "between_fixture_lux": between_fixture_lux,\n        "point_lux": point_lux,\n        "fixture_analysis": fixture_analysis,\n        "warnings": warning_texts,\n        "image_quality": image_quality,\n        "inference_skipped": False,\n        "floor_mask_ratio": floor_mask_ratio,\n    }\n    figure = save_result_artifacts(\n        output_dir=output_dir,\n        image_name=image_name,\n        display_rgb=display_rgb,\n        lux_map=lux_map,\n        fixture_analysis=fixture_analysis,\n        under_fixture_lux=under_fixture_lux,\n        between_fixture_lux=between_fixture_lux,\n        measurement_masks=measurement_masks,\n        result=result,\n        warning_texts=warning_texts,\n    )\n    return result, figure\n'
exec(HELPER_SOURCE, globals())

print(f"Project root: {PROJECT_ROOT}")
print(f"Default ONNX: {DEFAULT_ONNX_PATH}")
print(f"Output dir: {OUTPUT_DIR}")
print("Notebook preprocessing: ImageNet mean/std normalization enabled")
print(
    f"Dark-frame gate: mean<{DARK_FRAME_MEAN_THRESHOLD:.2f} and p95<{DARK_FRAME_P95_THRESHOLD:.2f} -> skip inference"
)
print("Fixture detector: corridor-geometry mode enabled")
print("Floor measurements: area-based floor sampling enabled")


In [ ]:
upload_widget = widgets.FileUpload(accept="image/*", multiple=False, description="Upload photo")
max_fixtures_widget = widgets.BoundedIntText(value=12, min=1, max=30, description="Max fixtures")
floor_area_widget = widgets.FloatText(value=12.0, description="Floor area m2")
run_button = widgets.Button(description="Run ONNX test", button_style="primary")
output_widget = widgets.Output()


def extract_uploaded_entry(upload_value):
    if isinstance(upload_value, dict) and upload_value:
        first_key = next(iter(upload_value.keys()))
        first_value = upload_value[first_key]
        if isinstance(first_value, dict):
            image_name = first_value.get("name", first_key)
            image_bytes = first_value.get("content")
            return image_name, image_bytes
        return str(first_key), first_value

    if isinstance(upload_value, (list, tuple)) and upload_value:
        first_value = upload_value[0]
        if isinstance(first_value, dict):
            image_name = first_value.get("name") or first_value.get("metadata", {}).get("name") or "uploaded_image"
            image_bytes = first_value.get("content")
            return image_name, image_bytes
        return "uploaded_image", first_value

    return None, None


def render_result(result: dict[str, Any]):
    display(Markdown(f"**Detected fixtures:** {result['fixture_count']}"))
    display(Markdown(f"**Average lux:** {result['avg_lux']:.1f}"))

    quality = result.get("image_quality") or {}
    if quality:
        print(
            "Image brightness: "
            f"mean={float(quality.get('mean_luminance', 0.0)):.3f}, "
            f"p95={float(quality.get('p95_luminance', 0.0)):.3f}, "
            f"bottom_mean={float(quality.get('bottom_mean_luminance', 0.0)):.3f}"
        )

    warnings = result.get("warnings") or []
    if warnings:
        print("Warnings:")
        for warning in warnings:
            print(f"  - {warning}")

    print("Floor lux directly under each fixture:")
    if result["under_fixture_lux"]:
        for point_name, lux_value in result["under_fixture_lux"].items():
            print(f"  - {point_name}: {lux_value:.1f} lux")
    else:
        print("  - none detected")

    print("Floor lux between adjacent fixtures:")
    if result["between_fixture_lux"]:
        for point_name, lux_value in result["between_fixture_lux"].items():
            print(f"  - {point_name}: {lux_value:.1f} lux")
    else:
        print("  - none detected")

    print(f"Overlay saved to: {result['overlay_image']}")
    print(f"Summary saved to: {result['summary_json']}")


def on_run_clicked(_):
    with output_widget:
        clear_output()
        image_name, image_bytes = extract_uploaded_entry(upload_widget.value)
        if image_bytes is None:
            print("Upload one image first.")
            return

        try:
            result, figure = run_uploaded_photo(
                image_bytes=image_bytes,
                image_name=image_name,
                onnx_path=DEFAULT_ONNX_PATH,
                output_dir=OUTPUT_DIR,
                max_fixture_count=int(max_fixtures_widget.value),
                floor_area_m2=float(floor_area_widget.value),
            )
        except Exception as exc:
            print(f"Inference failed: {exc}")
            return

        render_result(result)
        display(figure)
        plt.close(figure)


run_button.on_click(on_run_clicked)

display(
    widgets.VBox(
        [
            widgets.HTML("<b>Upload a hallway photo and run exported ONNX inference</b>"),
            widgets.HTML(f"<code>{DEFAULT_ONNX_PATH}</code>"),
            widgets.HTML(
                "Uses training-time ImageNet normalization and rejects near-black frames. "
                "Best results come from end-to-end hallway images with visible floor. Measurements are snapped onto the detected floor mask and averaged over floor areas."
            ),
            upload_widget,
            widgets.HBox([max_fixtures_widget, floor_area_widget]),
            run_button,
            output_widget,
        ]
    )
)
